In [ ]:
# =====================================================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# =====================================================================
# Installing LangChain, OpenAI tools, FAISS for vector DB, and PDF parsers.

!pip install -q langchain langchain-openai langchain-community faiss-cpu pypdf tiktoken
print("✅ All required libraries installed successfully inside the Colab environment!")

In [ ]:
# =====================================================================
# CELL 2: MOUNT GOOGLE DRIVE & EXTRACT PDFS
# =====================================================================
import os
from google.colab import drive

# 1. Mount Google Drive (Requires your permission)
print("🔄 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define Project Architecture Paths
PROJECT_DIR = "/content/drive/MyDrive/HM_RIE_Project"
KB_PATH = f"{PROJECT_DIR}/knowledge_base"
FAISS_DB_PATH = f"{PROJECT_DIR}/faiss_index"

# Create directories securely
os.makedirs(KB_PATH, exist_ok=True)
os.makedirs(FAISS_DB_PATH, exist_ok=True)

# 3. Extract the ZIP file containing your research papers
zip_path = "/content/drive/MyDrive/papers.zip"

if os.path.exists(zip_path):
    print(f"📦 Found 'papers.zip'. Extracting to {KB_PATH}...")
    # Clean the directory first to avoid overlaps
    !rm -rf {KB_PATH}/*
    # Unzip silently
    !unzip -q "{zip_path}" -d {KB_PATH}

    files = os.listdir(KB_PATH)
    print(f"✅ Extraction complete! Found {len(files)} items in the knowledge base.")
else:
    print("⚠️ 'papers.zip' not found in MyDrive. If you have already unzipped the files, you can ignore this warning.")

In [ ]:
# =====================================================================
# CELL 3: BUILD THE FAISS VECTOR DATABASE (THE AI BRAIN)
# =====================================================================
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# ⚠️ SECURE API KEY CONFIGURATION ⚠️
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"

print(f"📄 Scanning folder '{KB_PATH}' for PDF papers...")
loader = DirectoryLoader(KB_PATH, glob="**/*.pdf", loader_cls=PyPDFLoader)
docs = loader.load()

if len(docs) == 0:
    print("⚠️ No PDFs found! Please check your Google Drive folder.")
else:
    print(f"✅ Successfully loaded {len(docs)} pages from the literature.")

    # Chunking the documents (Size 1000, Overlap 200)
    print("✂️ Splitting text into semantic chunks to preserve scientific context...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)
    print(f"✅ Created {len(chunks)} text chunks.")

    # Creating Vector Embeddings and FAISS DB
    print("🧠 Creating FAISS Vector Database using OpenAI 'text-embedding-3-small'... (This may take a minute)")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vector_db = FAISS.from_documents(chunks, embeddings)

    # Save locally to Google Drive so it isn't lost when Colab closes
    vector_db.save_local(FAISS_DB_PATH)
    print(f"🎉 FAISS Index saved successfully at: {FAISS_DB_PATH}")

In [ ]:
# =====================================================================
# CELL 3: BUILD THE FAISS VECTOR DATABASE (THE AI BRAIN)
# =====================================================================
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# ⚠️ SECURE API KEY CONFIGURATION ⚠️
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"

print(f"📄 Scanning folder '{KB_PATH}' for PDF papers...")
loader = DirectoryLoader(KB_PATH, glob="**/*.pdf", loader_cls=PyPDFLoader)
docs = loader.load()

if len(docs) == 0:
    print("⚠️ No PDFs found! Please check your Google Drive folder.")
else:
    print(f"✅ Successfully loaded {len(docs)} pages from the literature.")

    # Chunking the documents (Size 1000, Overlap 200)
    print("✂️ Splitting text into semantic chunks to preserve scientific context...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)
    print(f"✅ Created {len(chunks)} text chunks.")

    # Creating Vector Embeddings and FAISS DB
    print("🧠 Creating FAISS Vector Database using OpenAI 'text-embedding-3-small'... (This may take a minute)")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vector_db = FAISS.from_documents(chunks, embeddings)

    # Save locally to Google Drive so it isn't lost when Colab closes
    vector_db.save_local(FAISS_DB_PATH)
    print(f"🎉 FAISS Index saved successfully at: {FAISS_DB_PATH}")